<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/temp/cross_section_momentum/ON%20ENTIRE%20EQUITY/equity_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
data_equity = pd.read_parquet('/content/drive/MyDrive/equities_combined.parquet')

In [3]:
data_equity.tail(5)

,SYMBOL,SERIES,OPEN,HIGH,LOW,CLOSE,LAST,PREVCLOSE,TOTTRDQTY,TOTTRDVAL,TIMESTAMP,TOTALTRADES,ISIN
3754602,ZYDUSWELL,EQ,386.70,389.10,378.10,384.90,387.85,384.85,257971,98796980.1,2026-02-27,10608,INE768C01028
3754603,ZYDUSWELL,EQ,367.60,384.15,367.55,378.95,380.00,384.90,172585,65482318.6,2026-03-02,11663,INE768C01028
3754604,ZYDUSWELL,EQ,378.40,382.50,371.30,377.10,376.80,378.95,131836,49849321.7,2026-03-04,11195,INE768C01028
3754605,ZYDUSWELL,EQ,379.25,385.90,375.10,383.00,383.00,377.10,77580,29456614.3,2026-03-05,8041,INE768C01028
3754606,ZYDUSWELL,EQ,381.00,386.60,380.00,381.20,380.55,383.00,67849,25972773.7,2026-03-06,7033,INE768C01028


In [4]:
data_equity.shape

(3754607, 13)

In [5]:
data_equity.columns

Index(['SYMBOL', 'SERIES', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'LAST', 'PREVCLOSE',
       'TOTTRDQTY', 'TOTTRDVAL', 'TIMESTAMP', 'TOTALTRADES', 'ISIN'],
      dtype='object')

In [6]:
data_equity.TIMESTAMP.max()


Timestamp('2026-03-06 00:00:00')

In [ ]:
import os
import zipfile
import shutil

source_folder = "/content"           # Folder containing ZIP files
levelone_folder = "/content/levelone"

os.makedirs(levelone_folder, exist_ok=True)

def extract_all_zips(source, destination):

    # First copy all zip files to levelone
    for file in os.listdir(source):
        if file.endswith(".zip"):
            shutil.copy(os.path.join(source, file), destination)

    # Now recursively extract inside levelone
    while True:
        zip_found = False

        for root, dirs, files in os.walk(destination):
            for file in files:
                if file.endswith(".zip"):
                    zip_found = True
                    zip_path = os.path.join(root, file)

                    with zipfile.ZipFile(zip_path, 'r') as z:
                        z.extractall(root)

                    os.remove(zip_path)
                    print(f"Extracted and removed: {file}")

        if not zip_found:
            break

    print("All ZIP files fully extracted into levelone.")


extract_all_zips(source_folder, levelone_folder)

In [8]:
import pandas as pd
import os

folder_path = "/content/levelone"

# Get all CSV files in the folder
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# Read and append
df_list = []

for file in csv_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    df_list.append(df)

# Combine all into one DataFrame
combined_df = pd.concat(df_list, ignore_index=True)

print(combined_df.shape)

(50309, 34)


In [9]:
cm_df = combined_df[combined_df['Sgmt'] == 'CM'].copy()

In [10]:
rename_map = {
    'TckrSymb': 'SYMBOL',
    'SctySrs': 'SERIES',
    'OpnPric': 'OPEN',
    'HghPric': 'HIGH',
    'LwPric': 'LOW',
    'ClsPric': 'CLOSE',
    'LastPric': 'LAST',
    'PrvsClsgPric': 'PREVCLOSE',
    'TtlTradgVol': 'TOTTRDQTY',
    'TtlTrfVal': 'TOTTRDVAL',
    'TradDt': 'TIMESTAMP',
    'TtlNbOfTxsExctd': 'TOTALTRADES'
}

In [11]:
cm_df = cm_df.rename(columns=rename_map)

In [12]:
required_cols = [
    'SYMBOL', 'SERIES', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'LAST',
    'PREVCLOSE', 'TOTTRDQTY', 'TOTTRDVAL', 'TIMESTAMP',
    'TOTALTRADES', 'ISIN'
]

cm_df = cm_df[required_cols]

In [13]:
cm_df['TIMESTAMP'] = pd.to_datetime(cm_df['TIMESTAMP'])
data_equity['TIMESTAMP'] = pd.to_datetime(data_equity['TIMESTAMP'])

In [14]:
final_df = pd.concat([data_equity, cm_df], ignore_index=True)

In [15]:
final_df = final_df.sort_values(['SYMBOL', 'TIMESTAMP'])

# Remove duplicates (keep latest)
final_df = final_df.drop_duplicates(
    subset=['SYMBOL', 'TIMESTAMP'],
    keep='last'
)

In [18]:
base_path = "/content/drive/MyDrive/data/partitioned"

In [ ]:
final_df['year'] = final_df['TIMESTAMP'].dt.year
final_df['month'] = final_df['TIMESTAMP'].dt.month

In [19]:
final_df.to_parquet(
    base_path,
    engine="pyarrow",
    compression="snappy",
    partition_cols=['year', 'month'],
    index=False
)